In [11]:
# SHA-as-Drift: full experiment suite per user's prompt blocks
# This notebook executes:
# 1) SHA_CALIBRATE
# 2) SHA_PLL_TUNER
# 3) SHA_COMPARATIVE
# 4) SHA_FEATURE_SET
# 5) BAND_ADJUDICATION
# 6) PREP_PI_ADDRESSING
# 7) PERSIST
#
# Artifacts are saved under d://Nexus//Nexus4//GTP5// and key tables are displayed.

import os, json, math, hashlib, time, statistics, io, zipfile
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Helpers ----------

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def msg_with_nonce(base: bytes, nonce: int) -> bytes:
    # Deterministic concatenation protocol
    return base + b":" + str(nonce).encode("utf-8")

def hex_to_bits(hex_str: str) -> np.ndarray:
    v = int(hex_str, 16)
    bits = np.array([(v >> (255 - i)) & 1 for i in range(256)], dtype=np.uint8)
    return bits

def leading_zero_bits(bits: np.ndarray) -> int:
    # Count from MSB until first 1
    # If all zero (impossible for sha256), returns 256
    z = 0
    for b in bits:
        if b == 0:
            z += 1
        else:
            break
    return z

def hamming_distance_bits(a: np.ndarray, b: np.ndarray) -> int:
    return int(np.bitwise_xor(a, b).sum())

def rolling_variance(x: pd.Series, window: int) -> pd.Series:
    return x.rolling(window=window, min_periods=window).var()

def acf(x: np.ndarray, nlags: int) -> np.ndarray:
    # Simple unbiased ACF up to nlags (excluding lag 0 return)
    if len(x) < nlags + 1:
        return np.full(nlags, np.nan)
    x = x - x.mean()
    denom = np.dot(x, x)
    if denom == 0:
        return np.zeros(nlags)
    res = []
    for k in range(1, nlags+1):
        res.append(np.dot(x[:-k], x[k:]) / denom)
    return np.array(res)

def contiguous_spans(mask: np.ndarray, start_nonce: int = 0) -> List[Tuple[int,int]]:
    spans = []
    n = len(mask)
    i = 0
    while i < n:
        if mask[i]:
            j = i
            while j+1 < n and mask[j+1]:
                j += 1
            spans.append((start_nonce + i, start_nonce + j))
            i = j + 1
        else:
            i += 1
    return spans

def save_plot(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    plt.close()

def glyph_id_from_manifest(manifest: Dict[str, Any]) -> str:
    s = json.dumps(manifest, sort_keys=True, separators=(",",":")).encode("utf-8")
    return hashlib.sha256(s).hexdigest()

# ---------- 1) SHA_CALIBRATE ----------

def sha_calibrate(base_msg: str,
                  nonce_range: Tuple[int,int],
                  H_target: float,
                  H_tolerance: float,
                  ham_var_window: int,
                  quiet_quantile: float,
                  out_prefix: str = "calibrate") -> Dict[str, Any]:
    base = base_msg.encode("utf-8")
    start, end = nonce_range
    N = end - start + 1
    rows = []
    prev_bits = None
    ham_list = []
    for i, nonce in enumerate(range(start, end+1)):
        h = sha256_hex(msg_with_nonce(base, nonce))
        bits = hex_to_bits(h)
        lz = leading_zero_bits(bits)
        if prev_bits is None:
            ham = np.nan
        else:
            ham = hamming_distance_bits(bits, prev_bits)
        prev_bits = bits
        ham_list.append(np.nan if math.isnan(ham) else ham)
        lz_norm = lz / 256.0
        drift_norm = (ham / 256.0) if not math.isnan(ham) else np.nan
        denom = (lz_norm + drift_norm) if not math.isnan(drift_norm) else np.nan
        H = (lz_norm / denom) if denom and not math.isnan(denom) and denom > 0 else np.nan
        rows.append((nonce, h, lz, ham, lz_norm, drift_norm, H))

    df = pd.DataFrame(rows, columns=["nonce","hash_hex","lz_bits","ham_to_prev","lz_norm","drift_norm","H"])
    df["ham_var"] = rolling_variance(df["ham_to_prev"], window=ham_var_window)

    # Define bands
    H_low, H_high = H_target - H_tolerance, H_target + H_tolerance
    df["in_H_band"] = df["H"].between(H_low, H_high)
    # Quiet threshold from available ham_var values
    valid_var = df["ham_var"].dropna()
    if len(valid_var) > 0:
        quiet_thresh = valid_var.quantile(quiet_quantile)
    else:
        quiet_thresh = float("inf")
    df["quiet"] = df["ham_var"] <= quiet_thresh

    # Accept both conditions and dwell >= 1 (simple spans)
    both = (df["in_H_band"] & df["quiet"]).fillna(False).to_numpy()
    spans = contiguous_spans(both, start_nonce=start)

    # Save artifacts
    csv_path = f"d://Nexus//Nexus4//GTP5//{out_prefix}_sha_phase_scan.csv"
    df.to_csv(csv_path, index=False)

    # Plots
    plt.figure(figsize=(10,3))
    plt.plot(df["nonce"], df["H"])
    plt.axhline(H_target, linestyle="--")
    plt.axhline(H_low, linestyle=":")
    plt.axhline(H_high, linestyle=":")
    save_plot(f"d://Nexus//Nexus4//GTP5//{out_prefix}_H_vs_nonce.png")

    plt.figure(figsize=(10,3))
    plt.plot(df["nonce"], df["ham_to_prev"])
    save_plot(f"d://Nexus//Nexus4//GTP5//{out_prefix}_hamming_vs_nonce.png")

    plt.figure(figsize=(10,3))
    plt.plot(df["nonce"], df["ham_var"])
    save_plot(f"d://Nexus//Nexus4//GTP5//{out_prefix}_ham_var_vs_nonce.png")

    summary = {
        "run": "SHA_CALIBRATE",
        "base_msg": base_msg,
        "nonce_range": [start, end],
        "H_target": H_target,
        "H_tolerance": H_tolerance,
        "ham_var_window": ham_var_window,
        "quiet_quantile": quiet_quantile,
        "quiet_threshold": quiet_thresh if quiet_thresh != float("inf") else None,
        "band_spans": spans,
        "artifacts": {
            "csv": csv_path,
            "plots": [
                f"d://Nexus//Nexus4//GTP5//{out_prefix}_H_vs_nonce.png",
                f"d://Nexus//Nexus4//GTP5//{out_prefix}_hamming_vs_nonce.png",
                f"d://Nexus//Nexus4//GTP5//{out_prefix}_ham_var_vs_nonce.png",
            ]
        }
    }
    with open(f"d://Nexus//Nexus4//GTP5//{out_prefix}_summary.json","w") as f:
        json.dump(summary, f, indent=2)
    return {"df": df, "summary": summary}

# ---------- 2) SHA_PLL_TUNER ----------

def sha_pll_tuner(base_msg: str,
                  max_steps: int,
                  init_nonce: int,
                  init_step: int,
                  alpha: float,
                  beta: float,
                  ham_var_window: int,
                  quiet_quantile: float,
                  step_floor: int,
                  step_ceiling: int,
                  stop_H_band: Tuple[float,float],
                  stop_quiet: bool,
                  stop_dwell: int,
                  out_prefix: str = "pll") -> Dict[str, Any]:

    base = base_msg.encode("utf-8")

    def metrics(nonce: int, prev_bits: Optional[np.ndarray], ham_hist: List[int]) -> Dict[str, Any]:
        h = sha256_hex(msg_with_nonce(base, nonce))
        bits = hex_to_bits(h)
        if prev_bits is None:
            ham = np.nan
        else:
            ham = hamming_distance_bits(bits, prev_bits)
        lz = leading_zero_bits(bits)
        lz_norm = lz / 256.0
        drift_norm = (ham / 256.0) if not math.isnan(ham) else np.nan
        denom = (lz_norm + drift_norm) if not math.isnan(drift_norm) else np.nan
        H = (lz_norm / denom) if denom and not math.isnan(denom) and denom > 0 else np.nan

        ham_hist2 = ham_hist + ([] if math.isnan(ham) else [ham])
        var = np.nan
        if len(ham_hist2) >= ham_var_window:
            series = pd.Series(ham_hist2[-ham_var_window:])
            var = float(series.var())
        return {"nonce": nonce, "hash_hex": h, "bits": bits, "ham": ham, "H": H, "ham_var": var, "lz_bits": lz}

    # Initialization
    cur_nonce = init_nonce
    prev_bits = None
    ham_hist: List[int] = []
    step = init_step
    path = []
    # For dynamic quiet threshold we keep a global pool of ham_var values (ignoring NaN)
    ham_vars_all: List[float] = []

    H_lo, H_hi = stop_H_band
    dwell_counter = 0
    best_spans: List[Tuple[int,int]] = []
    in_band = False
    band_start = None

    # Evaluate initial
    m = metrics(cur_nonce, prev_bits, ham_hist)
    prev_bits = m["bits"]
    if not math.isnan(m["ham"]):
        ham_hist.append(int(m["ham"]))
    if not math.isnan(m["ham_var"]):
        ham_vars_all.append(float(m["ham_var"]))
    path.append(m)

    def objective(H: float, ham_var: float, quiet_thresh: Optional[float]) -> float:
        # Penalize NaNs heavily
        if H is None or math.isnan(H):
            return 1e9
        var_pen = 0.0
        if ham_var is not None and not math.isnan(ham_var):
            var_pen = ham_var
            # optionally scale by quiet threshold (not necessary)
        return alpha * (H - 0.35)**2 + beta * var_pen

    for t in range(1, max_steps+1):
        # Dynamic quiet threshold
        quiet_thresh = None
        if len(ham_vars_all) >= 32:
            quiet_thresh = float(pd.Series(ham_vars_all).quantile(quiet_quantile))

        # Try forward
        cand_plus = metrics(cur_nonce + step, prev_bits, ham_hist)
        # Try backward (ensure non-negative nonce)
        back_nonce = max(0, cur_nonce - step)
        cand_minus = metrics(back_nonce, prev_bits, ham_hist)

        cur = path[-1]
        cur_obj = objective(cur["H"], cur["ham_var"], quiet_thresh)
        obj_plus = objective(cand_plus["H"], cand_plus["ham_var"], quiet_thresh)
        obj_minus = objective(cand_minus["H"], cand_minus["ham_var"], quiet_thresh)

        # Choose best move
        chosen = None
        if obj_plus < cur_obj or obj_minus < cur_obj:
            chosen = cand_plus if obj_plus <= obj_minus else cand_minus
            cur_nonce = chosen["nonce"]
            prev_bits = chosen["bits"]
            if not math.isnan(chosen["ham"]):
                ham_hist.append(int(chosen["ham"]))
            if not math.isnan(chosen["ham_var"]):
                ham_vars_all.append(float(chosen["ham_var"]))
            path.append(chosen)
        else:
            # Flip direction and reduce step (damped)
            step = max(step_floor, step // 2)
            step = min(step, step_ceiling)
            # If step is minimal and no improvement, take a small random nudge to escape
            if step == step_floor:
                cur_nonce = max(0, cur_nonce + (1 if (t % 2 == 0) else -1))
                chosen = metrics(cur_nonce, prev_bits, ham_hist)
                prev_bits = chosen["bits"]
                if not math.isnan(chosen["ham"]):
                    ham_hist.append(int(chosen["ham"]))
                if not math.isnan(chosen["ham_var"]):
                    ham_vars_all.append(float(chosen["ham_var"]))
                path.append(chosen)
            else:
                # Continue without move recorded (rare)
                pass

        # Stopping / band tracking
        last = path[-1]
        in_H = (last["H"] is not None and not math.isnan(last["H"]) and (H_lo <= last["H"] <= H_hi))
        is_quiet = False
        if stop_quiet and quiet_thresh is not None and last["ham_var"] is not None and not math.isnan(last["ham_var"]):
            is_quiet = (last["ham_var"] <= quiet_thresh)
        elif not stop_quiet:
            is_quiet = True

        if in_H and is_quiet:
            dwell_counter += 1
            if not in_band:
                in_band = True
                band_start = last["nonce"]
        else:
            if in_band:
                best_spans.append((band_start, path[-2]["nonce"] if len(path)>=2 else band_start))
            in_band = False
            dwell_counter = 0

        if dwell_counter >= stop_dwell:
            # close current band
            if in_band:
                best_spans.append((band_start, last["nonce"]))
            break

        # Grow step slightly when improving
        if len(path) >= 3:
            if objective(path[-1]["H"], path[-1]["ham_var"], quiet_thresh) < objective(path[-2]["H"], path[-2]["ham_var"], quiet_thresh):
                step = min(step_ceiling, step * 2)

    # Save trajectory
    traj = pd.DataFrame([
        {"nonce": p["nonce"], "hash_hex": p["hash_hex"], "lz_bits": p["lz_bits"],
         "ham_to_prev": p["ham"], "H": p["H"], "ham_var": p["ham_var"]}
        for p in path
    ])
    traj_csv = f"d://Nexus//Nexus4//GTP5//{out_prefix}_trajectory.csv"
    traj.to_csv(traj_csv, index=False)

    # Plot trajectory
    plt.figure(figsize=(10,3))
    plt.plot(traj["nonce"], traj["H"])
    plt.axhline(0.35, linestyle="--")
    plt.axhline(H_lo, linestyle=":")
    plt.axhline(H_hi, linestyle=":")
    save_plot(f"d://Nexus//Nexus4//GTP5//{out_prefix}_H_trajectory.png")

    plt.figure(figsize=(10,3))
    plt.plot(traj["nonce"], traj["ham_to_prev"])
    save_plot(f"d://Nexus//Nexus4//GTP5//{out_prefix}_ham_trajectory.png")

    summary = {
        "run": "SHA_PLL_TUNER",
        "base_msg": base_msg,
        "steps_taken": int(len(traj)),
        "final_nonce": int(traj["nonce"].iloc[-1]),
        "final_H": float(traj["H"].iloc[-1]) if not math.isnan(traj["H"].iloc[-1]) else None,
        "best_spans": best_spans,
        "artifacts": {
            "trajectory_csv": traj_csv,
            "plots": [f"d://Nexus//Nexus4//GTP5//{out_prefix}_H_trajectory.png",
                      f"d://Nexus//Nexus4//GTP5//{out_prefix}_ham_trajectory.png"]
        }
    }
    with open(f"d://Nexus//Nexus4//GTP5//{out_prefix}_summary.json","w") as f:
        json.dump(summary, f, indent=2)

    return {"traj": traj, "summary": summary}

# ---------- 3) SHA_COMPARATIVE ----------

def sha_comparative(base_A: str,
                    base_B: str,
                    nonce_range: Tuple[int,int],
                    H_target: float,
                    H_tolerance: float,
                    ham_var_window: int,
                    quiet_quantile: float,
                    coherence_rule: str,
                    out_prefix: str = "comparative") -> Dict[str, Any]:

    # Reuse calibrate for each base
    A = sha_calibrate(base_A, nonce_range, H_target, H_tolerance, ham_var_window, quiet_quantile, out_prefix=f"{out_prefix}_A")
    B = sha_calibrate(base_B, nonce_range, H_target, H_tolerance, ham_var_window, quiet_quantile, out_prefix=f"{out_prefix}_B")
    dfA, dfB = A["df"], B["df"]

    # Align indices by nonce
    merged = dfA.merge(dfB, on="nonce", suffixes=("_A","_B"))
    both = (merged["in_H_band_A"] & merged["quiet_A"] & merged["in_H_band_B"] & merged["quiet_B"]).to_numpy()
    spans = contiguous_spans(both, start_nonce=int(merged["nonce"].iloc[0]) if not merged.empty else 0)

    # Export overlays
    plt.figure(figsize=(10,3))
    plt.plot(merged["nonce"], merged["H_A"], label="H_A")
    plt.plot(merged["nonce"], merged["H_B"], label="H_B")
    plt.axhline(H_target, linestyle="--")
    plt.axhline(H_target-H_tolerance, linestyle=":")
    plt.axhline(H_target+H_tolerance, linestyle=":")
    plt.legend()
    save_plot(f"d://Nexus//Nexus4//GTP5//{out_prefix}_H_overlay.png")

    summary = {
        "run": "SHA_COMPARATIVE",
        "base_A": base_A,
        "base_B": base_B,
        "nonce_range": list(nonce_range),
        "coherent_spans": spans,
        "artifacts": {
            "overlay_plot": f"d://Nexus//Nexus4//GTP5//{out_prefix}_H_overlay.png",
            "A_csv": A["summary"]["artifacts"]["csv"],
            "B_csv": B["summary"]["artifacts"]["csv"]
        }
    }
    with open(f"d://Nexus//Nexus4//GTP5//{out_prefix}_summary.json","w") as f:
        json.dump(summary, f, indent=2)

    return {"merged": merged, "summary": summary}

# ---------- 4) SHA_FEATURE_SET ----------

def walsh_hadamard_energy(bits: np.ndarray, k: int) -> float:
    # Compute energy at frequency index k via Walsh–Hadamard (simple transform on +/-1)
    x = 1 - 2*bits.astype(np.int8)  # map {0,1} -> {+1,-1}
    # Fast Walsh–Hadamard Transform
    h = x.astype(np.int32)
    n = len(h)
    # next power of two (should be 256 already)
    m = 1
    while m < n:
        for i in range(0, n, m*2):
            for j in range(i, i+m):
                u = h[j]; v = h[j+m]
                h[j] = u + v
                h[j+m] = u - v
        m *= 2
    # Normalize
    return float(h[k]**2) / float(n*n)

def prefix_runlen_bits(bits: np.ndarray, max_check: int = 64) -> int:
    # Longest initial run (from MSB) where all bits equal first bit
    first = bits[0]
    cnt = 1
    for b in bits[1:max_check]:
        if b == first:
            cnt += 1
        else:
            break
    return cnt

def hex_prefix_zero_nibbles(hex_str: str) -> int:
    cnt = 0
    for ch in hex_str:
        if ch == '0':
            cnt += 1
        else:
            break
    return cnt

def sha_feature_set(base_msg: str,
                    nonce_range: Tuple[int,int],
                    weights: Dict[str,float],
                    compose: str,
                    validate_bootstrap: int,
                    out_prefix: str = "features") -> Dict[str, Any]:

    base = base_msg.encode("utf-8")
    start, end = nonce_range
    rows = []
    for nonce in range(start, end+1):
        h = sha256_hex(msg_with_nonce(base, nonce))
        bits = hex_to_bits(h)
        f_lz = leading_zero_bits(bits) / 256.0
        f_wht = walsh_hadamard_energy(bits, k=4)
        f_run = prefix_runlen_bits(bits) / 256.0
        f_nib = hex_prefix_zero_nibbles(h) / 64.0  # 64 hex chars
        rows.append((nonce, h, f_lz, f_wht, f_run, f_nib))

    df = pd.DataFrame(rows, columns=["nonce","hash_hex","lz_bits","wht_energy_k4","prefix_runlen_bits","hex_prefix_zero_nibbles"])
    # Normalize features to [0,1] rough scale where needed (wht energy already [0,1])
    # Compose H*
    w = weights
    # Already scaled as per construction
    H_star = (w["lz_bits"]*df["lz_bits"] +
              w["wht_energy_k=4"]*df["wht_energy_k4"] +
              w["prefix_runlen_bits"]*df["prefix_runlen_bits"] +
              w["hex_prefix_zero_nibbles"]*df["hex_prefix_zero_nibbles"]) / sum(w.values())
    df["H_star"] = H_star

    csv_path = f"d://Nexus//Nexus4//GTP5//{out_prefix}_feature_scan.csv"
    df.to_csv(csv_path, index=False)

    # Stability via bootstrap: resample nonces and report std of H_star mean
    rng = np.random.default_rng(42)
    means = []
    n = len(df)
    for _ in range(validate_bootstrap):
        idx = rng.integers(0, n, size=n)
        means.append(df["H_star"].to_numpy()[idx].mean())
    stats = {"H_star_mean": float(df["H_star"].mean()),
             "H_star_std": float(df["H_star"].std()),
             "bootstrap_mean_std": float(np.std(means))}

    plt.figure(figsize=(10,3))
    plt.plot(df["nonce"], df["H_star"])
    plt.axhline(0.35, linestyle="--")
    save_plot(f"d://Nexus//Nexus4//GTP5//{out_prefix}_Hstar_vs_nonce.png")

    summary = {
        "run": "SHA_FEATURE_SET",
        "base_msg": base_msg,
        "nonce_range": list(nonce_range),
        "weights": weights,
        "stats": stats,
        "artifacts": {
            "csv": csv_path,
            "plot": f"d://Nexus//Nexus4//GTP5//{out_prefix}_Hstar_vs_nonce.png"
        }
    }
    with open(f"d://Nexus//Nexus4//GTP5//{out_prefix}_summary.json","w") as f:
        json.dump(summary, f, indent=2)
    return {"df": df, "summary": summary}

# ---------- 5) BAND_ADJUDICATION ----------

def band_adjudication(calibrate_csv_path: str,
                      ham_var_window: int,
                      ac_window: int,
                      criteria: Dict[str, Any],
                      dwell_min: int,
                      out_prefix: str = "bands") -> Dict[str, Any]:

    df = pd.read_csv(calibrate_csv_path)
    # Ensure rolling var exists
    if "ham_var" not in df.columns or df["ham_var"].isna().all():
        df["ham_var"] = rolling_variance(df["ham_to_prev"], window=ham_var_window)

    # Quiet threshold
    quiet_thresh = df["ham_var"].dropna().quantile(0.20) if df["ham_var"].notna().any() else float("inf")
    quiet_mask = df["ham_var"] <= quiet_thresh

    # H band
    H_low, H_high = criteria["H_band"]
    H_mask = df["H"].between(H_low, H_high)

    # Stationarity via ΔACF over sliding windows on ham_to_prev
    ham = df["ham_to_prev"].to_numpy()
    n = len(ham)
    acf_change = np.full(n, np.nan)
    prev_vec = None
    for i in range(n):
        L = max(0, i - ac_window + 1)
        window = ham[L:i+1]
        window = window[~np.isnan(window)]
        if len(window) >= ac_window//2:
            vec = acf(window.astype(float), nlags=8)
            if prev_vec is not None and not np.any(np.isnan(vec)) and not np.any(np.isnan(prev_vec)):
                acf_change[i] = float(np.linalg.norm(vec - prev_vec))
            prev_vec = vec
        else:
            prev_vec = None
            acf_change[i] = np.nan

    df["acf_delta"] = acf_change
    # Stationary threshold
    if np.isfinite(df["acf_delta"]).any():
        stat_thresh = np.nanquantile(df["acf_delta"], 0.10)
    else:
        stat_thresh = float("inf")
    stationary_mask = df["acf_delta"] <= stat_thresh

    both = (quiet_mask & H_mask & stationary_mask).fillna(False).to_numpy()
    spans_all = contiguous_spans(both, start_nonce=int(df["nonce"].iloc[0]))
    # Enforce dwell_min
    accepted = []
    rejected = []
    for s,e in spans_all:
        if (e - s + 1) >= dwell_min:
            accepted.append((s,e))
        else:
            rejected.append({"span": [s,e], "reason": f"dwell<{dwell_min}"})

    # Save
    acc_path = f"d://Nexus//Nexus4//GTP5//{out_prefix}_accepted_bands.json"
    rej_path = f"d://Nexus//Nexus4//GTP5//{out_prefix}_rejected_bands.json"
    with open(acc_path,"w") as f:
        json.dump({"accepted_bands": accepted,
                   "quiet_threshold": None if quiet_thresh==float('inf') else float(quiet_thresh),
                   "stationary_threshold": None if stat_thresh==float('inf') else float(stat_thresh)}, f, indent=2)
    with open(rej_path,"w") as f:
        json.dump({"rejected": rejected}, f, indent=2)

    summary = {
        "run": "BAND_ADJUDICATION",
        "criteria": criteria,
        "dwell_min": dwell_min,
        "accepted_bands": accepted,
        "artifacts": {
            "accepted_bands": acc_path,
            "rejected_bands": rej_path
        }
    }
    with open(f"d://Nexus//Nexus4//GTP5//{out_prefix}_summary.json","w") as f:
        json.dump(summary, f, indent=2)

    return {"df": df, "summary": summary}

# ---------- 6) PREP_PI_ADDRESSING ----------

def prep_pi_addressing(base_msg: str,
                       calibrate_df: pd.DataFrame,
                       accepted_bands: List[Tuple[int,int]],
                       export_path: str = "d://Nexus//Nexus4//GTP5//pi_address_candidates.jsonl") -> str:
    # Representative nonce: midpoint of span
    lines = []
    for (s,e) in accepted_bands:
        rep = (s + e) // 2
        # Recover hash at representative nonce from df (if present), else recompute
        row = calibrate_df[calibrate_df["nonce"] == rep]
        if not row.empty:
            hhex = str(row["hash_hex"].iloc[0])
        else:
            hhex = sha256_hex(msg_with_nonce(base_msg.encode("utf-8"), rep))
        record = {
            "base_msg": base_msg,
            "nonce_span": [int(s), int(e)],
            "rep_nonce": int(rep),
            "hash_hex": hhex
        }
        lines.append(record)

    with open(export_path,"w") as f:
        for r in lines:
            f.write(json.dumps(r) + "\n")
    return export_path

# ---------- 7) PERSIST ----------

def persist_bundle(manifest_entries: List[str],
                   out_dir: str = "d://Nexus//Nexus4//GTP5//",
                   bundle_name: str = "sha_drift_bundle") -> Dict[str, Any]:
    # Create manifest with file paths and sha256 checksums
    entries = []
    for p in manifest_entries:
        if not os.path.exists(p):
            continue
        with open(p,"rb") as f:
            data = f.read()
        entries.append({"path": p, "sha256": hashlib.sha256(data).hexdigest(), "bytes": len(data)})
    manifest = {"created": time.time(), "entries": entries}
    manifest_path = os.path.join(out_dir, "manifest.json")
    with open(manifest_path,"w") as f:
        json.dump(manifest, f, indent=2)

    # Glyph id as sha256(manifest)
    gid = glyph_id_from_manifest(manifest)
    with open(os.path.join(out_dir,"glyph_ids.json"),"w") as f:
        json.dump({"manifest_sha256": gid}, f, indent=2)

    # Write a human-readable run summary text
    with open(os.path.join(out_dir,"run_sha256.txt"),"w") as f:
        f.write("Bundle entries (path, sha256, bytes)\n")
        for e in entries:
            f.write(f"{e['path']}  {e['sha256']}  {e['bytes']}\n")
        f.write(f"\nManifest glyph id: {gid}\n")

    # Zip bundle
    zip_path = os.path.join(out_dir, f"{bundle_name}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(manifest_path, arcname="manifest.json")
        z.write(os.path.join(out_dir,"glyph_ids.json"), arcname="glyph_ids.json")
        z.write(os.path.join(out_dir,"run_sha256.txt"), arcname="run_sha256.txt")
        for e in entries:
            rel = os.path.basename(e["path"])
            z.write(e["path"], arcname=rel)

    return {"manifest": manifest_path, "glyph_id": gid, "zip": zip_path}

# ----------------- Execute per user's parameters -----------------

# Parameters from the prompt blocks
base_msg = "PGGSPHRKCGYDLQNRGHPQW"
nonce_range = (0, 50000)
H_target = 0.35
H_tolerance = 0.02
ham_var_window = 33
quiet_quantile = 0.20

# 1) Calibrate
calib = sha_calibrate(base_msg, nonce_range, H_target, H_tolerance, ham_var_window, quiet_quantile, out_prefix="calibrate")

# 2) PLL tuner
pll = sha_pll_tuner(base_msg, max_steps=30000, init_nonce=0, init_step=17,
                    alpha=1.0, beta=0.5, ham_var_window=ham_var_window, quiet_quantile=quiet_quantile,
                    step_floor=1, step_ceiling=4096,
                    stop_H_band=(0.33, 0.37), stop_quiet=True, stop_dwell=64, out_prefix="pll")

# 3) Comparative (two-track)
compar = sha_comparative(base_A=base_msg,
                         base_B=base_msg + "\x00",
                         nonce_range=nonce_range,
                         H_target=H_target, H_tolerance=H_tolerance,
                         ham_var_window=ham_var_window, quiet_quantile=quiet_quantile,
                         coherence_rule="retain spans where BOTH tracks satisfy H-band AND quiet",
                         out_prefix="comparative")

# 4) Feature expansion scan (use first 10k nonces to bound runtime)
feat = sha_feature_set(base_msg, nonce_range=(0, 10000),
                       weights={"lz_bits":0.40, "wht_energy_k=4":0.20, "prefix_runlen_bits":0.20, "hex_prefix_zero_nibbles":0.20},
                       compose="H = sum(w_i * f_i) / sum(w_i)",
                       validate_bootstrap=200,
                       out_prefix="features")

# 5) Band adjudication using calibrate CSV
bands = band_adjudication(calibrate_csv_path=calib["summary"]["artifacts"]["csv"],
                          ham_var_window=ham_var_window, ac_window=64,
                          criteria={"H_band":[0.33,0.37]}, dwell_min=64, out_prefix="bands")

# 6) PI address preparation
pi_addr_path = prep_pi_addressing(base_msg, calib["df"], bands["summary"]["accepted_bands"],
                                  export_path="d://Nexus//Nexus4//GTP5//pi_address_candidates.jsonl")

# 7) Persist
files_to_bundle = [
    calib["summary"]["artifacts"]["csv"],
    *calib["summary"]["artifacts"]["plots"],
    "d://Nexus//Nexus4//GTP5//calibrate_summary.json",
    pll["summary"]["artifacts"]["trajectory_csv"],
    *pll["summary"]["artifacts"]["plots"],
    "d:/Nexus/Nexus4/GTP5/pll_summary.json",
    compar["summary"]["artifacts"]["overlay_plot"],
    "d://Nexus//Nexus4//GTP5//comparative_A_sha_phase_scan.csv",
    "d://Nexus//Nexus4//GTP5//comparative_B_sha_phase_scan.csv",
    "d://Nexus//Nexus4//GTP5//comparative_summary.json",
    feat["summary"]["artifacts"]["csv"],
    feat["summary"]["artifacts"]["plot"],
    "d://Nexus//Nexus4//GTP5//features_summary.json",
    "d://Nexus//Nexus4//GTP5//bands_accepted_bands.json",
    "d://Nexus//Nexus4//GTP5//bands_rejected_bands.json",
    "d://Nexus//Nexus4//GTP5//bands_summary.json",
    pi_addr_path
]
persist = persist_bundle(files_to_bundle, out_dir="d://Nexus//Nexus4//GTP5//", bundle_name="sha_drift_bundle")

# Display key tables to user
from caas_jupyter_tools import display_dataframe_to_user
display_dataframe_to_user("Calibrate (first 200 rows)", calib["df"].head(200))
display_dataframe_to_user("PLL Trajectory", pd.read_csv(pll["summary"]["artifacts"]["trajectory_csv"]))
display_dataframe_to_user("Comparative (merged head)", compar["merged"].head(200))
display_dataframe_to_user("Feature scan (head)", feat["df"].head(200))

# Print quick links
print("Artifacts:")
print("Calibrate CSV:", calib["summary"]["artifacts"]["csv"])
print("Calibrate plots:", calib["summary"]["artifacts"]["plots"])
print("PLL trajectory:", pll["summary"]["artifacts"]["trajectory_csv"])
print("Comparative overlay:", compar["summary"]["artifacts"]["overlay_plot"])
print("Feature scan CSV:", feat["summary"]["artifacts"]["csv"])
print("Band adjudication accepted:", "d://Nexus//Nexus4//GTP5//bands_accepted_bands.json")
print("PI address candidates:", pi_addr_path)
print("Bundle zip:", persist["zip"])
print("Manifest glyph id:", persist["glyph_id"])


ModuleNotFoundError: No module named 'caas_jupyter_tools'

In [12]:
# sha_calibrate.py
# Deterministic SHA-256 nonce sweep → H-index, rolling variance, quiet bands.
# Artifacts: CSV + plots + summary JSON.

import json, math, hashlib
from typing import Tuple, List, Dict, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- low-level bits ----------

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def msg_with_nonce(base: bytes, nonce: int) -> bytes:
    # fixed concatenation protocol: base:decimal_nonce
    return base + b":" + str(nonce).encode("utf-8")

def hex_to_bits(hex_str: str) -> np.ndarray:
    v = int(hex_str, 16)
    # MSB-first, length 256
    return np.array([(v >> (255 - i)) & 1 for i in range(256)], dtype=np.uint8)

def leading_zero_bits(bits: np.ndarray) -> int:
    z = 0
    for b in bits:
        if b == 0: z += 1
        else: break
    return z

def hamming_distance_bits(a: np.ndarray, b: np.ndarray) -> int:
    return int(np.bitwise_xor(a, b).sum())

def rolling_variance(x: pd.Series, window: int) -> pd.Series:
    return x.rolling(window=window, min_periods=window).var()

def contiguous_spans(mask: np.ndarray, start_nonce: int = 0) -> List[Tuple[int,int]]:
    spans = []
    n = len(mask); i = 0
    while i < n:
        if mask[i]:
            j = i
            while j+1 < n and mask[j+1]:
                j += 1
            spans.append((start_nonce + i, start_nonce + j))
            i = j + 1
        else:
            i += 1
    return spans

def save_plot(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    plt.close()

# ---------- H-index (delta-first) ----------

def sweep_calibrate(base_msg: str,
                    nonce_range: Tuple[int,int],
                    H_target: float = 0.35,
                    H_tolerance: float = 0.02,
                    ham_var_window: int = 33,
                    quiet_quantile: float = 0.20,
                    out_prefix: str = "calibrate") -> Dict[str, Any]:

    base = base_msg.encode("utf-8")
    start, end = nonce_range

    rows = []
    prev_bits = None
    for nonce in range(start, end+1):
        h = sha256_hex(msg_with_nonce(base, nonce))
        bits = hex_to_bits(h)
        lz = leading_zero_bits(bits)
        ham = float("nan") if prev_bits is None else hamming_distance_bits(bits, prev_bits)
        prev_bits = bits

        lz_norm = lz / 256.0
        drift_norm = (ham / 256.0) if not math.isnan(ham) else float("nan")
        denom = (lz_norm + drift_norm) if not math.isnan(drift_norm) else float("nan")
        H = (lz_norm / denom) if denom and not math.isnan(denom) and denom > 0 else float("nan")

        rows.append((nonce, h, lz, ham, lz_norm, drift_norm, H))

    df = pd.DataFrame(rows, columns=["nonce","hash_hex","lz_bits","ham_to_prev","lz_norm","drift_norm","H"])
    df["ham_var"] = rolling_variance(df["ham_to_prev"], window=ham_var_window)

    H_low, H_high = H_target - H_tolerance, H_target + H_tolerance
    df["in_H_band"] = df["H"].between(H_low, H_high)

    valid_var = df["ham_var"].dropna()
    quiet_thresh = valid_var.quantile(quiet_quantile) if len(valid_var) > 0 else float("inf")
    df["quiet"] = df["ham_var"] <= quiet_thresh

    both = (df["in_H_band"] & df["quiet"]).fillna(False).to_numpy()
    spans = contiguous_spans(both, start_nonce=start)

    csv_path = f"{out_prefix}_sha_phase_scan.csv"
    df.to_csv(csv_path, index=False)

    # H-index over nonce
    plt.figure(figsize=(10,3))
    plt.plot(df["nonce"], df["H"])
    plt.axhline(H_target, linestyle="--")
    plt.axhline(H_low, linestyle=":")
    plt.axhline(H_high, linestyle=":")
    save_plot(f"{out_prefix}_H_vs_nonce.png")

    # Hamming-to-previous
    plt.figure(figsize=(10,3))
    plt.plot(df["nonce"], df["ham_to_prev"])
    save_plot(f"{out_prefix}_hamming_vs_nonce.png")

    # Rolling variance
    plt.figure(figsize=(10,3))
    plt.plot(df["nonce"], df["ham_var"])
    save_plot(f"{out_prefix}_ham_var_vs_nonce.png")

    summary = {
        "run": "SHA_CALIBRATE",
        "base_msg": base_msg,
        "nonce_range": [start, end],
        "H_target": H_target,
        "H_tolerance": H_tolerance,
        "ham_var_window": ham_var_window,
        "quiet_quantile": quiet_quantile,
        "quiet_threshold": None if quiet_thresh == float("inf") else float(quiet_thresh),
        "band_spans": spans,
        "artifacts": {
            "csv": csv_path,
            "plots": [
                f"{out_prefix}_H_vs_nonce.png",
                f"{out_prefix}_hamming_vs_nonce.png",
                f"{out_prefix}_ham_var_vs_nonce.png",
            ]
        }
    }
    with open(f"{out_prefix}_summary.json","w") as f:
        json.dump(summary, f, indent=2)

    # Console certificate
    print(json.dumps({
        "ps": "calibrate-complete",
        "H_band": [H_low, H_high],
        "quiet_q": quiet_quantile,
        "quiet_threshold": summary["quiet_threshold"],
        "first_spans": spans[:5]
    }, indent=2))

    return summary

if __name__ == "__main__":
    # Defaults from your prompt
    sweep_calibrate(
        base_msg="PGGSPHRKCGYDLQNRGHPQW",
        nonce_range=(0, 50000),   # you can lower to (0, 20000) for a quick run
        H_target=0.35,
        H_tolerance=0.02,
        ham_var_window=33,
        quiet_quantile=0.20,
        out_prefix="calibrate"
    )


{
  "ps": "calibrate-complete",
  "H_band": [
    0.32999999999999996,
    0.37
  ],
  "quiet_q": 0.2,
  "quiet_threshold": 50.5795454545453,
  "first_spans": []
}
